# 02 — CartPole Experiment

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Mattral/rssmlite/blob/main/notebooks/02_cartpole_experiment.ipynb)

Full training run of `rssmlite` on `CartPole-v1`. Expected runtime on a free
Colab T4: **~20 minutes** for 200 000 environment steps.

**P1.3 exit-criterion check:** mean return ≥ 195 over 20 eval episodes.


In [ ]:
# ── Bootstrap ─────────────────────────────────────────────────────────────
!pip install -q "rssmlite[envs,viz] @ git+https://github.com/Mattral/rssmlite.git"
from google.colab import drive
drive.mount("/content/drive")

import torch
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
print(f"Device: {gpu}")
assert torch.cuda.is_available(), "Switch runtime: Runtime → Change runtime type → T4 GPU"


In [ ]:
import os
if not os.path.exists("/content/rssmlite"):
    !git clone -q https://github.com/Mattral/rssmlite.git /content/rssmlite


In [ ]:
import torch, gymnasium as gym, time, re
import numpy as np
from rssmlite import RSSMAgent

CHECKPOINT_DIR = "/content/drive/MyDrive/rssmlite-ckpts/cartpole"
DEVICE = torch.device("cuda")

env = gym.make("CartPole-v1")

agent = RSSMAgent.from_config("/content/rssmlite/configs/cartpole.yaml", env=env)
agent.to(DEVICE)   # moves rssm, actor, critic in one call — no monkey-patching needed
print(f"Agent on {agent.device}. Starting training...")


In [ ]:
# ── Training loop ──────────────────────────────────────────────────────────
episode_rewards, env_steps_log = [], []

def log_fn(msg):
    m = re.match(r"\[step (\d+)\] episode_reward=([\d.\-]+)", msg)
    if m:
        env_steps_log.append(int(m.group(1)))
        episode_rewards.append(float(m.group(2)))
    if len(episode_rewards) % 20 == 0:
        print(msg)

t0 = time.time()
agent.train(
    env,
    steps=200_000,
    checkpoint_dir=CHECKPOINT_DIR,
    checkpoint_every=10_000,
    log_every=1,
    log_fn=log_fn,
)
print(f"\nTraining complete in {(time.time()-t0)/60:.1f} min")
env.close()


In [ ]:
# ── Training curve ─────────────────────────────────────────────────────────
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

def smooth(x, w=20):
    return np.convolve(x, np.ones(w)/w, mode="valid")

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(env_steps_log, episode_rewards, alpha=0.3, color="steelblue", label="raw")
if len(episode_rewards) >= 20:
    ax.plot(env_steps_log[19:], smooth(episode_rewards),
            color="steelblue", linewidth=2, label="smoothed (w=20)")
ax.axhline(195, color="tomato", linestyle="--", label="solve threshold (195)")
ax.set_xlabel("environment steps")
ax.set_ylabel("episode return")
ax.set_title("rssmlite on CartPole-v1")
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("cartpole_training_curve.png", dpi=150)
plt.show()
print(f"Final 20-ep average: {np.mean(episode_rewards[-20:]):.1f}")


In [ ]:
# ── Reconstruction quality ─────────────────────────────────────────────────
from rssmlite import reconstruction_report
from rssmlite.evaluation import plot_reconstruction

batch = agent.buffer.sample(batch_size=32, seq_len=50)
report = reconstruction_report(agent, batch)   # _batch_to_device handled internally
print("Reconstruction MSE (symlog space):", f"{report['mse_symlog']:.4f}")
print("Reconstruction MSE (real units)  :", f"{report['mse_real']:.4f}")
print("Per-feature MSE:", report["per_dim_mse"].cpu().numpy().round(4))
print("(CartPole dims: cart_pos, cart_vel, pole_angle, pole_angular_vel)")


In [ ]:
# ── Real vs predicted observation plot ────────────────────────────────────
batch2 = agent.buffer.sample(batch_size=4, seq_len=50)
fig = plot_reconstruction(agent, batch2, feature_idx=2)  # pole angle
plt.savefig("cartpole_reconstruction.png", dpi=150)
plt.show()


In [ ]:
# ── Imagined rollout ───────────────────────────────────────────────────────
rollout = agent.imagine_rollout(steps=15)
print("Imagined rewards over 15 steps:")
print(rollout["reward_pred"].squeeze().cpu().detach().numpy().round(3))


In [ ]:
# ── Latent space ──────────────────────────────────────────────────────────
fig = agent.visualize_latent_space()
plt.savefig("cartpole_latent_space.png", dpi=150)
plt.show()


In [ ]:
# ── Evaluation episodes ────────────────────────────────────────────────────
from rssmlite import run_evaluation_episodes

eval_env = gym.make("CartPole-v1")
results = run_evaluation_episodes(agent, eval_env, n_episodes=20)
eval_env.close()

print(f"Evaluation over 20 episodes:")
print(f"  Mean return : {results['mean_return']:.1f}")
print(f"  Std return  : {results['std_return']:.1f}")
print(f"  Min / Max   : {results['min_return']:.1f} / {results['max_return']:.1f}")
print(f"  Solved (>=195) : {results['mean_return'] >= 195}")


## Resuming after a disconnect

```python
import glob
from rssmlite import RSSMAgent
import torch, gymnasium as gym

CHECKPOINT_DIR = "/content/drive/MyDrive/rssmlite-ckpts/cartpole"
ckpts = sorted(glob.glob(f"{CHECKPOINT_DIR}/checkpoint_*.pt"))
print("Latest checkpoint:", ckpts[-1])

env = gym.make("CartPole-v1")
agent = RSSMAgent.load_checkpoint(ckpts[-1])
agent.to(torch.device("cuda"))
print(f"Resuming from step {agent._env_steps}")

agent.train(env, steps=200_000, checkpoint_dir=CHECKPOINT_DIR)
```

See `docs/colab_guide.md` for the full recovery flow.
